# Securities Analysis - Volatility

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexeyklek10/Security_Analysis/blob/main/notebooks/Volatility_Analysis.ipynb)

Per-asset OHLC volatility analysis using seven estimators (Realized,
Parkinson, Garman-Klass, Rogers-Satchell, Yang-Zhang, EWMA RiskMetrics,
GARCH(1,1) MLE), bias-corrected Hurst (Anis-Lloyd 1976), Lo-MacKinlay
variance ratio test at four lags, and a heavy-tail event diagnostic that
flags suspect days for the more exotic tickers.

## How to read this notebook

- Each ticker gets a 4-panel dashboard saved to `images/volatility/`.
- Console output mirrors the dashboards in tabular form, with regime
  classification, Hurst regime, and per-test significance flags.
- The cross-asset section ends with a top-5 |log_return| days diagnostic
  for known heavy-tail tickers so the reader can sanity-check that the
  reported kurtosis comes from real events rather than vendor artifacts.

## Setup


In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
  INSTITUTIONAL VOLATILITY ANALYTICS SUITE v2.0
  Multi-Asset | Multi-Estimator | Statistical Testing | Regime Detection
================================================================================

  Designed for portfolio managers, risk analysts, and quantitative researchers.
  Provides comprehensive volatility decomposition with institutional-grade
  statistical testing and actionable regime interpretation.

  Run in Google Colab or any Python 3.9+ environment with:
      pip install yfinance pandas numpy matplotlib seaborn scipy statsmodels

  Author:  Custom Build
  Version: 2.0
================================================================================
"""

# ==============================================================================
#  SECTION 1: IMPORTS & SYSTEM INITIALIZATION
# ==============================================================================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.lines import Line2D
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats as sps
from scipy.stats import jarque_bera, kurtosis, skew
import warnings
import sys
import traceback
import hashlib
from pathlib import Path

warnings.filterwarnings('ignore')


# F35: discover the repo root regardless of whether the notebook is run from
# Analysis_securities/ or Analysis_securities/notebooks/. We walk up until
# we hit either requirements.txt or a .git directory; failing both, we fall
# back to the current working directory (typical Colab case).
def _find_repo_root(start: Path = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (cur / 'requirements.txt').exists() or (cur / '.git').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return Path.cwd().resolve()

REPO_ROOT = _find_repo_root()


# ==============================================================================
#  SECTION 2: THEME & STYLING
# ==============================================================================

# --- Dark Institutional Theme ---
THEME = {
    # Core palette
    'bg':           '#0e1117',
    'panel':        '#161b22',
    'grid':         '#21262d',
    'text':         '#c9d1d9',
    'text_dim':     '#8b949e',
    'text_bright':  '#f0f6fc',
    'accent':       '#58a6ff',

    # Semantic colors
    'blue':         '#58a6ff',
    'cyan':         '#39d2c0',
    'green':        '#3fb950',
    'yellow':       '#d29922',
    'orange':       '#db6d28',
    'red':          '#f85149',
    'purple':       '#bc8cff',
    'pink':         '#f778ba',

    # Volatility estimator colors (consistent across all charts)
    'realized':     '#58a6ff',
    'parkinson':    '#39d2c0',
    'garman_klass': '#3fb950',
    'yang_zhang':   '#d29922',
    'ewma':         '#bc8cff',
    'garch':        '#f85149',
    'downside':     '#f778ba',

    # Regime colors
    'regime_low':   '#238636',
    'regime_mid':   '#9e6a03',
    'regime_high':  '#da3633',
    'regime_ext':   '#8b0000',

    # Fonts
    'font_title':   13,
    'font_label':   10,
    'font_tick':    8,
    'font_legend':  8,
    'font_annot':   9,
}

def apply_theme():
    """Apply the dark institutional theme globally."""
    plt.rcParams.update({
        'figure.facecolor':     THEME['bg'],
        'axes.facecolor':       THEME['panel'],
        'axes.edgecolor':       THEME['grid'],
        'axes.labelcolor':      THEME['text'],
        'axes.grid':            True,
        'grid.color':           THEME['grid'],
        'grid.alpha':           0.5,
        'grid.linewidth':       0.5,
        'text.color':           THEME['text'],
        'xtick.color':          THEME['text_dim'],
        'ytick.color':          THEME['text_dim'],
        'xtick.labelsize':      THEME['font_tick'],
        'ytick.labelsize':      THEME['font_tick'],
        'legend.facecolor':     THEME['panel'],
        'legend.edgecolor':     THEME['grid'],
        'legend.fontsize':      THEME['font_legend'],
        'legend.framealpha':    0.9,
        'figure.dpi':           110,
        'savefig.dpi':          150,
        'font.family':          'monospace',
        'axes.titlesize':       THEME['font_title'],
        'axes.labelsize':       THEME['font_label'],
    })

apply_theme()


# ==============================================================================


## Configuration

`TICKERS` is the universe (29 ETFs spanning precious metals, miners,
uranium, Japan, AI/robotics, utilities, alternatives, treasuries,
international). `SHOWCASE` (= GLD) gets a deeper-dive treatment.

GARCH mode is `'mle'` by default (fitted per asset via `arch.arch_model`);
swap to `'manual'` for a pedagogical fixed-parameter recursion (see the
comment block inside `garch_volatility`).


In [ ]:
#  SECTION 3: CONFIGURATION PANEL
# ==============================================================================

# --- ASSET UNIVERSE (modify as needed) ---
TICKERS = [
    # ── Precious Metals ──
    'GLD',    # Gold (SPDR)
    'SLV',    # Silver (iShares)
    'PPLT',   # Platinum (abrdn)

    # ── Miners ──
    'GDX',    # Gold Miners (VanEck)
    'COPX',   # Copper Miners (Global X)

    # ── Uranium ──
    'URA',    # Uranium broad (Global X)
    'URNM',   # Uranium Miners (Sprott)

    # ── Japan ──
    'EWJ',    # Japan Large Cap (iShares MSCI)
    'DFJ',    # Japan Small Cap (WisdomTree)

    # ── AI / Future Tech ──
    'BOTZ',   # Robotics & AI (Global X)
    'AIQ',    # AI & Big Data (Global X)
    'ROBT',   # Nasdaq AI & Robotics (First Trust)

    # ── Utilities ──
    'XLU',    # Utilities Select Sector

    # ── Alternatives ──
    'DBMF',   # Managed Futures (iM DBi)
    'MNA',    # Merger Arbitrage (NYLI / IQ)
    'BTAL',   # Anti-Beta / Market Neutral (AGFiQ)
    'ILS',    # Insurance-Linked Securities / Cat Bond (Stone Ridge)

    # ── Treasuries (duration ladder) ──
    'SHY',    # 1-3 Year Treasury
    'IEI',    # 3-7 Year Treasury
    'IEF',    # 7-10 Year Treasury
    'TLH',    # 10-20 Year Treasury
    'TLT',    # 20+ Year Treasury
    'TIP',    # TIPS (inflation-linked)

    # ── EM Equity ──
    'INDA',   # India (iShares MSCI)
    'VNM',    # Vietnam (VanEck)

    # ── Real Assets ──
    'VNQI',   # International REITs (Vanguard)
    'GNR',    # Global Natural Resources (SPDR)

    # ── Commodities (enhanced roll) ──
    'CERY.L', # SPDR Bloomberg Enhanced Roll Yield (LSE)

    # ── Single Name ──
    'RPRX',   # Royalty Pharma
]

# --- DATE RANGE (10-year lookback for robust statistics) ---
START_DATE = '2015-01-01'
END_DATE   = '2026-01-01'
MIN_OBSERVATIONS = 252  # Minimum 1 year of data to run analysis

# --- ROLLING WINDOWS ---
ROLLING_WINDOW_SHORT  = 10    # ~2 weeks  (fast / tactical)
ROLLING_WINDOW_MEDIUM = 30    # ~1 month  (standard)
ROLLING_WINDOW_LONG   = 63    # ~1 quarter (strategic)

# --- EWMA (RiskMetrics) ---
EWMA_LAMBDA = 0.94

# --- GARCH(1,1) Manual Calibration ---
GARCH_MODE = 'mle'   # 'mle' (per-asset MLE via arch package, recommended) or
                     # 'manual' (legacy fixed-alpha/beta recursion; pedagogical only).
                     # See garch_volatility() docstring.

# --- DOWNSIDE DEVIATION ---
DOWNSIDE_MAR = 0.0   # Minimum Acceptable Return (annualized)

# --- ATR ---
ATR_PERIOD = 14

# --- HURST EXPONENT ---
HURST_MAX_LAG = 100
HURST_WINDOW  = 120   # Longer window for stability

# --- VaR / CVaR ---
VAR_CONFIDENCE = 0.95
VAR_WINDOW     = 252   # 1 year rolling

# --- VARIANCE RATIO TEST ---
VR_LAGS = [2, 5, 10, 20]

# --- ANNUALIZATION ---
TRADING_DAYS = 252

# --- DISPLAY / OUTPUT ---
SHOW_SUMMARY         = True
SHOW_CROSS_ASSET     = True
SAVE_FIGURES         = True
VOLATILITY_IMAGE_DIR = REPO_ROOT / 'images' / 'volatility'
CACHE_DIR            = 'data/cache'   # parquet price cache (F19)


# ==============================================================================
#  SECTION 4: CONSOLE HEADER
# ==============================================================================

def print_header():
    """Print startup banner with configuration summary."""
    w = 74
    print(f"\n{'━' * w}")
    print(f"{'INSTITUTIONAL VOLATILITY ANALYTICS SUITE v2.0':^{w}}")
    print(f"{'━' * w}")
    print(f"  {'Assets':<22} {len(TICKERS)} tickers")
    print(f"  {'Universe':<22} {', '.join(TICKERS[:6])}{'…' if len(TICKERS) > 6 else ''}")
    print(f"  {'Period':<22} {START_DATE}  →  {END_DATE}  (~{int((pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days/365)}y)")
    print(f"  {'Rolling Windows':<22} {ROLLING_WINDOW_SHORT} / {ROLLING_WINDOW_MEDIUM} / {ROLLING_WINDOW_LONG} days")
    print(f"  {'EWMA λ':<22} {EWMA_LAMBDA}")
    print(f"  {'GARCH mode':<22} {GARCH_MODE}  (parameters fitted per asset via MLE)")
    print(f"  {'VaR Confidence':<22} {VAR_CONFIDENCE:.0%}")
    print(f"  {'Min Observations':<22} {MIN_OBSERVATIONS}")
    print(f"{'━' * w}\n")


# ==============================================================================


## Data fetch

Per-ticker yfinance download via the F19 parquet cache. The function
applies two vendor-error filters before computing returns:

1. A general rule: drop any row where `Volume == 0` and `Close` differs
   from the previous `Close` by more than 0.5%. With no trades the
   close should equal the prior close; a phantom change on a zero-
   volume day is a known yfinance stub-row artifact.
2. A named mask for BTAL on 2015-04-29 (a $12.96 print inside a $19.30
   corridor on volume 109k that survives the general filter). Details
   are documented inside `fetch_ohlc_data`.


In [ ]:
#  SECTION 5: DATA ENGINE
# ==============================================================================

# F19: Lazy parquet cache. Keyed on the (ticker, start, end) triple via a
# short stable hash so cache files stay readable in a directory listing.
# Cache hits skip the yfinance round-trip entirely; cache misses fetch from
# yfinance and persist the OHLCV frame as parquet. The cache directory is
# created on first use. Both notebooks contain this helper verbatim so each
# remains copy-pasteable into a single Colab cell.
CACHE_DIR = REPO_ROOT / 'data' / 'cache'


def _cache_key(ticker: str, start: str, end: str) -> str:
    h = hashlib.sha1(f'{ticker}|{start}|{end}'.encode('utf-8')).hexdigest()[:10]
    safe_ticker = ticker.replace('.', '_').replace('/', '_').replace('^', '_')
    return f'{safe_ticker}_{start}_{end}_{h}.parquet'


def _cached_yf_download(ticker: str, start: str, end: str,
                        auto_adjust: bool = True, timeout: int = 30):
    """yfinance.download wrapped with a per-(ticker, start, end) parquet cache.

    Cache hits return a pandas DataFrame indexed by date with the same
    OHLCV columns the rest of the notebook expects. Misses fetch from
    yfinance, normalize to flat columns, write parquet, and return.
    """
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = CACHE_DIR / _cache_key(ticker, start, end)
    if cache_path.exists():
        try:
            df = pd.read_parquet(cache_path)
            if df is not None and not df.empty:
                return df
        except Exception:
            pass  # corrupted cache → re-fetch
    raw = yf.download(ticker, start=start, end=end, progress=False,
                      auto_adjust=auto_adjust, timeout=timeout)
    if raw is None or raw.empty:
        return raw
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [col[0] if isinstance(col, tuple) else col for col in raw.columns]
    try:
        raw.to_parquet(cache_path)
    except Exception:
        pass  # cache-write failures are non-fatal
    return raw


def fetch_ohlc_data(ticker, start, end):
    """
    Fetch OHLC data from Yahoo Finance.

    Uses auto_adjust=True so Open/High/Low/Close are all split- and dividend-
    adjusted consistently (F17). This is essential for OHLC-range volatility
    estimators: a mismatch between adjusted Close and unadjusted O/H/L produces
    spurious jumps on dividend / split days that bleed into kurtosis, the
    Jarque-Bera test, and every range-based estimator.

    Returns a clean DataFrame with columns:
        Open, High, Low, Close, Volume,
        Log_Return, Simple_Return, Overnight_Return, Intraday_Return
    """
    try:
        # F19: parquet cache wraps the yfinance round-trip. The helper
        # also flattens MultiIndex columns on miss so callers always see
        # the same simple shape.
        raw = _cached_yf_download(ticker, start=start, end=end,
                                  auto_adjust=True, timeout=30)

        if raw is None or raw.empty:
            print(f"    ✗ No data returned for {ticker}")
            return pd.DataFrame()

        required = ['Open', 'High', 'Low', 'Close', 'Volume']
        missing = [c for c in required if c not in raw.columns]
        if missing:
            print(f"    ✗ Missing columns for {ticker}: {missing}")
            return pd.DataFrame()

        data = pd.DataFrame(index=raw.index)
        data['Open']   = pd.to_numeric(raw['Open'], errors='coerce')
        data['High']   = pd.to_numeric(raw['High'], errors='coerce')
        data['Low']    = pd.to_numeric(raw['Low'], errors='coerce')
        data['Close']  = pd.to_numeric(raw['Close'], errors='coerce')
        data['Volume'] = pd.to_numeric(raw['Volume'], errors='coerce')

        data = data.ffill(limit=5).dropna(subset=['Open', 'High', 'Low', 'Close'])
        data = data[(data[['Open', 'High', 'Low', 'Close']] > 0).all(axis=1)]

        # ── Vendor-error masks ─────────────────────────────────────────────
        # Two stacked filters address Yahoo data artifacts that
        # auto_adjust=True cannot reach.
        #
        # Filter 1 — general: drop any row where Volume==0 AND Close
        # differs from the previous Close by more than 0.5%. With zero
        # trades, the recorded close should equal the previous close;
        # a phantom price change on a no-trade day is by construction
        # a vendor artifact (typically a ffill/stub-row glitch). The
        # 0.5% threshold is set above the largest observed legitimate
        # adjustment drift and below the smallest observed artifact
        # (1%). This filter catches:
        #
        #   BTAL 2014-12-30   Close=$17.00 on Vol=0  (was ~$17.91)
        #   BTAL 2014-12-31   Close=$17.91 (snap-back) on Vol=0
        #   BTAL 2015-04-30   Close=$19.29 (snap-back from 2015-04-29) on Vol=0
        #   DBMF 2019-09-16, 2020-01-31, 2020-02-25, 2020-08-04 on Vol=0
        #
        # Filter 2 — BTAL 2015-04-29: a bad-print row with Volume=109k,
        # Close=$12.96 inside a ~$19.30 corridor. The same pattern
        # appears with auto_adjust=False, so it is not an adjustment
        # issue. Because volume is nonzero, the general filter cannot
        # catch it; it has to be named explicitly. Documented in
        # PARTIAL_CHANGELOG.md (Phase A owner halt + autonomy-charter
        # follow-up sweep across all volatility tickers).
        prev_close = data['Close'].shift(1)
        bad_zero_vol = (
            (data['Volume'] == 0)
            & prev_close.notna()
            & ((data['Close'] / prev_close - 1).abs() > 0.005)
        )
        data = data.loc[~bad_zero_vol]

        if ticker == 'BTAL':
            data = data.loc[~data.index.isin([pd.Timestamp('2015-04-29')])]

        if len(data) < MIN_OBSERVATIONS:
            print(f"    ✗ {ticker}: only {len(data)} observations (need {MIN_OBSERVATIONS})")
            return pd.DataFrame()

        data['Log_Return']       = np.log(data['Close'] / data['Close'].shift(1))
        data['Simple_Return']    = data['Close'].pct_change()
        data['Overnight_Return'] = np.log(data['Open'] / data['Close'].shift(1))
        data['Intraday_Return']  = np.log(data['Close'] / data['Open'])

        return data.dropna()

    except Exception as e:
        print(f"    ✗ ERROR fetching {ticker}: {e}")
        return pd.DataFrame()


# ==============================================================================


## Volatility estimators

The seven estimators in one section. Each closed-form formula lives in
the function docstring; headline relationships:

- **Realized** (close-to-close): the baseline, efficiency 1x.
- **Parkinson** (high-low): ~5x more efficient than RV, downward-biased
  when drift is nonzero.
- **Garman-Klass** (OHLC): 7-8x efficient; drift-sensitive.
- **Rogers-Satchell**: drift-robust OHLC.
- **Yang-Zhang**: GK + overnight gap; designed for gappy series.
- **EWMA**: RiskMetrics-style with lambda = 0.94.
- **GARCH(1,1) MLE**: per-asset, via `arch_model(... vol='Garch')`.

All rolling estimators use `min_periods = window` so the series starts
honestly at the first index with a full lookback (F15).


In [ ]:
#  SECTION 6: VOLATILITY ESTIMATORS
# ==============================================================================

def realized_volatility(returns, window, annualize=True):
    """
    Close-to-close realized volatility.
    Most common; ignores intraday information.
    Efficiency: 1x (baseline).
    """
    vol = returns.rolling(window=window, min_periods=window).std()
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def parkinson_volatility(high, low, window, annualize=True):
    """
    Parkinson (1980) high-low range estimator.
    Uses intraday range → ~5x more efficient than close-to-close.
    Downward bias when drift is nonzero or prices are discrete.
    """
    log_hl = np.log(high / low)
    factor = 1.0 / (4.0 * np.log(2))
    var_daily = factor * (log_hl ** 2)
    vol = np.sqrt(var_daily.rolling(window=window, min_periods=window).mean())
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def garman_klass_volatility(open_p, high, low, close, window, annualize=True):
    """
    Garman-Klass (1980) OHLC estimator.
    Uses full OHLC bar → ~7.4x more efficient than close-to-close.
    Assumes no drift and continuous prices.
    """
    log_hl = np.log(high / low)
    log_co = np.log(close / open_p)
    var_daily = 0.5 * (log_hl ** 2) - (2.0 * np.log(2) - 1.0) * (log_co ** 2)
    vol = np.sqrt(
        var_daily.rolling(window=window, min_periods=window).mean().clip(lower=0)
    )
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def rogers_satchell_volatility(open_p, high, low, close, window, annualize=True):
    """
    Rogers-Satchell (1991) estimator.
    Allows for nonzero drift — more robust than Garman-Klass
    when assets are trending.
    """
    log_ho = np.log(high / open_p)
    log_hc = np.log(high / close)
    log_lo = np.log(low / open_p)
    log_lc = np.log(low / close)
    var_daily = log_ho * log_hc + log_lo * log_lc
    vol = np.sqrt(
        var_daily.rolling(window=window, min_periods=window).mean().clip(lower=0)
    )
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def yang_zhang_volatility(open_p, high, low, close, window, annualize=True):
    """
    Yang-Zhang (2000) — the gold-standard OHLC estimator.
    Handles overnight jumps, drift, and opening gaps.
    Combines overnight variance, open-to-close variance,
    and Rogers-Satchell variance with optimal weighting.
    """
    log_co = np.log(open_p / close.shift(1))
    log_oc = np.log(close / open_p)
    log_ho = np.log(high / open_p)
    log_lo = np.log(low / open_p)
    log_hc = np.log(high / close)
    log_lc = np.log(low / close)

    rs = log_ho * log_hc + log_lo * log_lc
    k = 0.34 / (1.34 + (window + 1) / (window - 1))

    # F15: min_periods = window so the vol series starts honestly at
    # the first index where the full lookback window is available;
    # earlier indices remain NaN instead of being half-fit estimates.
    min_p = window
    var_overnight  = log_co.rolling(window=window, min_periods=min_p).var()
    var_open_close = log_oc.rolling(window=window, min_periods=min_p).var()
    var_rs         = rs.rolling(window=window, min_periods=min_p).mean()

    variance = (var_overnight + k * var_open_close + (1 - k) * var_rs).clip(lower=0)
    vol = np.sqrt(variance)
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def ewma_volatility(returns, lambda_param, annualize=True):
    """
    EWMA (Exponentially Weighted Moving Average) — RiskMetrics approach.
    No fixed window: recent observations weighted more heavily.
    λ = 0.94 is the J.P. Morgan RiskMetrics daily standard.
    Higher λ → smoother; Lower λ → more reactive.
    """
    sq_ret = returns ** 2
    ewma_var = sq_ret.ewm(alpha=(1 - lambda_param), adjust=False).mean()
    vol = np.sqrt(ewma_var)
    return vol * np.sqrt(TRADING_DAYS) if annualize else vol


def garch_volatility(returns, annualize=True, mode='mle'):
    """
    GARCH(1,1) conditional volatility.

    mode='mle' (default, F04): parameters estimated per-asset by maximum
        likelihood via `arch.arch_model`. Returns the conditional volatility
        series and a dict with fitted (alpha, beta, omega, persistence,
        half_life). On convergence failure, returns (NaN series, dict of NaNs).

    mode='manual': fixed-parameter recursion (legacy / pedagogical). See the
        commented `_manual_garch` reference implementation below — kept in the
        codebase as documentation but no longer called in the default path.

    Persistence = α + β; must be < 1 for stationarity. Half-life of a vol
    shock = log(0.5) / log(α + β).

    Reference for manual recursion (not executed):
        # def _manual_garch(returns, alpha=0.08, beta=0.90):
        #     n = len(returns); var = np.zeros(n); lr = returns.var()
        #     omega = max(lr * (1 - alpha - beta), 1e-12); var[0] = lr
        #     r = returns.values
        #     for t in range(1, n):
        #         var[t] = max(omega + alpha*r[t-1]**2 + beta*var[t-1], 1e-12)
        #     return pd.Series(np.sqrt(var), index=returns.index)
    """
    from arch import arch_model

    clean = returns.dropna()
    if len(clean) < 100:
        nan_series = pd.Series(np.nan, index=returns.index)
        return nan_series, {'alpha': np.nan, 'beta': np.nan, 'omega': np.nan,
                            'persistence': np.nan, 'half_life': np.nan,
                            'converged': False}

    try:
        # arch expects returns on a percent-ish scale for numerical stability
        am = arch_model(clean * 100, vol='Garch', p=1, q=1,
                        mean='Zero', dist='normal', rescale=False)
        res = am.fit(disp='off', show_warning=False)
        a = float(res.params.get('alpha[1]', np.nan))
        b = float(res.params.get('beta[1]', np.nan))
        w = float(res.params.get('omega', np.nan))
        persistence = a + b
        half_life = np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.nan
        cond_vol_pct = res.conditional_volatility
        cond_vol_daily = cond_vol_pct / 100.0
        cond_vol = cond_vol_daily.reindex(returns.index)
        params = {'alpha': a, 'beta': b, 'omega': w,
                  'persistence': persistence, 'half_life': half_life,
                  'converged': bool(res.convergence_flag == 0)}
        return (cond_vol * np.sqrt(TRADING_DAYS) if annualize else cond_vol), params
    except Exception as e:
        print(f"    ⚠ GARCH MLE failed: {e}")
        nan_series = pd.Series(np.nan, index=returns.index)
        return nan_series, {'alpha': np.nan, 'beta': np.nan, 'omega': np.nan,
                            'persistence': np.nan, 'half_life': np.nan,
                            'converged': False}


# ==============================================================================
#  SECTION 7: RISK METRICS & STATISTICAL TESTS
# ==============================================================================

def downside_deviation(returns, mar=0, window=None, annualize=True):
    """
    Downside deviation (semi-deviation below MAR).
    Core input to the Sortino ratio.
    """
    daily_mar = mar / TRADING_DAYS
    downside = np.minimum(returns - daily_mar, 0)

    if window:
        sq = (downside ** 2).rolling(window=window, min_periods=window).mean()
        dd = np.sqrt(sq)
    else:
        dd = np.sqrt((downside ** 2).mean())

    return dd * np.sqrt(TRADING_DAYS) if annualize else dd


def volatility_of_volatility(vol_series, window):
    """Vol-of-vol: measures regime stability. Spikes → regime transitions."""
    return vol_series.rolling(window=window, min_periods=window).std()


def calculate_atr(high, low, close, period=14):
    """
    Average True Range — Wilder's smoothed version.
    Measures full daily range including overnight gaps.
    Primary input for position sizing.
    """
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()


def atr_percentile(atr_series, lookback=252):
    """Where current ATR sits vs. trailing 1-year distribution."""
    def _pct(x):
        if len(x) < 20:
            return 50.0
        return sps.percentileofscore(x[:-1], x.iloc[-1])
    return atr_series.rolling(window=lookback, min_periods=60).apply(_pct, raw=False)


def _expected_rs(n):
    """
    Anis-Lloyd (1976) expected R/S(n) under iid normal null. Used to bias-correct
    the finite-sample R/S Hurst estimator (F05).

    Closed form:
        E[R/S(n)] = (n - 1/2) / n * G(n) * sum_{r=1..n-1} sqrt((n-r)/r)
    where for n ≤ 340: G(n) = Γ((n-1)/2) / (√π · Γ(n/2))
    and for n > 340 the Stirling asymptote G(n) ≈ sqrt(2 / (n·π)) is used.
    """
    if n < 2:
        return np.nan
    r = np.arange(1, n)
    s = float(np.sum(np.sqrt((n - r) / r)))
    if n <= 340:
        from math import lgamma
        log_g = lgamma((n - 1) / 2.0) - 0.5 * np.log(np.pi) - lgamma(n / 2.0)
        return (n - 0.5) / n * np.exp(log_g) * s
    return (n - 0.5) / n * np.sqrt(2.0 / (n * np.pi)) * s


def calculate_hurst_exponent(returns, max_lag=100, bias_correct=True):
    """
    Hurst exponent via rescaled-range (R/S) analysis.

      H < 0.5 → mean-reverting (anti-persistent)
      H = 0.5 → random walk
      H > 0.5 → trending (persistent)

    With bias_correct=True (default, F05) the Anis-Lloyd (1976) finite-sample
    correction is applied: subtract the expected R/S-slope of an iid process
    from the empirical slope, then recenter at 0.5. On pure Gaussian noise
    this brings the estimator's expectation back to ≈ 0.5 (the raw R/S
    estimator has a ≈ +0.08 upward bias for n ≈ 2500).

    No clip to [0, 1] is applied (F16); extreme slopes are diagnostically
    useful and the caller can flag them.
    """
    ret = returns.dropna().values
    n = len(ret)
    if n < 40:
        return np.nan

    max_k = min(max_lag, n // 4)
    if max_k < 8:
        return np.nan

    rs_list, lag_list = [], []
    for lag in range(8, max_k):
        num_sub = n // lag
        if num_sub < 1:
            continue
        rs_sub = []
        for i in range(num_sub):
            sub = ret[i * lag:(i + 1) * lag]
            if len(sub) < 4:
                continue
            mean_adj = sub - np.mean(sub)
            cum = np.cumsum(mean_adj)
            R = np.max(cum) - np.min(cum)
            S = np.std(sub, ddof=1)
            if S > 1e-10 and R > 0:
                rs_sub.append(R / S)
        if rs_sub:
            rs_list.append(np.mean(rs_sub))
            lag_list.append(lag)

    if len(rs_list) < 8:
        return np.nan

    log_lags = np.log(np.array(lag_list))
    log_rs = np.log(np.array(rs_list))
    valid = np.isfinite(log_lags) & np.isfinite(log_rs)
    if np.sum(valid) < 8:
        return np.nan

    h_empirical, _, _, _, _ = sps.linregress(log_lags[valid], log_rs[valid])

    if not bias_correct:
        return h_empirical

    expected = np.array([_expected_rs(int(lag)) for lag in lag_list])
    log_exp = np.log(expected)
    valid2 = valid & np.isfinite(log_exp)
    if np.sum(valid2) < 8:
        return h_empirical
    h_expected, _, _, _, _ = sps.linregress(log_lags[valid2], log_exp[valid2])
    return h_empirical - h_expected + 0.5


def rolling_hurst(returns, window, max_lag=50):
    """Rolling Anis-Lloyd-corrected Hurst exponent over a fixed window."""
    vals, idx = [], []
    for i in range(window, len(returns)):
        h = calculate_hurst_exponent(returns.iloc[i - window:i],
                                     max_lag=max_lag, bias_correct=True)
        vals.append(h)
        idx.append(returns.index[i])
    return pd.Series(vals, index=idx, dtype=float)


def market_efficiency_ratio(rv, gk):
    """RV / GK ratio: >1 → overnight-driven; <1 → intraday-driven."""
    ratio = rv / gk.replace(0, np.nan)
    return ratio.replace([np.inf, -np.inf], np.nan)


def rolling_var_cvar(returns, window=252, confidence=0.95):
    """
    Historical Value-at-Risk and Conditional VaR (Expected Shortfall).
    Non-parametric — uses empirical quantile of the return distribution.
    """
    alpha = 1 - confidence
    var_series = returns.rolling(window=window, min_periods=60).quantile(alpha)

    def _cvar(x):
        cutoff = np.quantile(x, alpha)
        tail = x[x <= cutoff]
        return tail.mean() if len(tail) > 0 else cutoff

    cvar_series = returns.rolling(window=window, min_periods=60).apply(_cvar, raw=True)
    return var_series, cvar_series


def variance_ratio_test(returns, lag=2):
    """
    Lo-MacKinlay (1988) variance ratio test, overlapping-returns form.

    VR ≈ 1 → random walk;  VR > 1 → positive serial corr (momentum-like);
    VR < 1 → negative serial corr (mean reversion).

    Asymptotic SE under homoscedasticity (Campbell-Lo-MacKinlay 1997 eq. 2.4.39):
        Var(VR(k) - 1) = 2 (2k - 1)(k - 1) / (3 k T)
    The earlier (pre-F03) implementation used the non-overlapping-returns SE
    2(k-1)/(3kT), which produced over-large z-statistics and false positives
    against the random-walk null.

    Returns (VR, z-statistic, p-value).
    """
    ret = returns.dropna().values
    n = len(ret)
    if n < lag * 4:
        return np.nan, np.nan, np.nan

    mu = np.mean(ret)
    var_1 = np.sum((ret - mu) ** 2) / (n - 1)
    # k-period overlapping sums of returns
    ret_k = np.convolve(ret, np.ones(lag), mode='valid')
    var_k = np.sum((ret_k - lag * mu) ** 2) / (n - lag)

    if var_1 == 0:
        return np.nan, np.nan, np.nan

    vr = var_k / (lag * var_1)
    se = np.sqrt(2 * (2 * lag - 1) * (lag - 1) / (3 * lag * n))
    z = (vr - 1) / se if se > 0 else 0
    p = 2 * (1 - sps.norm.cdf(abs(z)))
    return vr, z, p


def return_distribution_tests(returns):
    """
    Battery of distribution tests on log returns.
    Returns dict with test names → (statistic, p-value, interpretation).
    """
    ret = returns.dropna().values
    results = {}

    # Jarque-Bera normality test
    jb_stat, jb_p = jarque_bera(ret)
    results['Jarque-Bera'] = {
        'stat': jb_stat, 'p': jb_p,
        'interpretation': 'Non-normal (fat tails / skew)' if jb_p < 0.05 else 'Consistent with normal'
    }

    # Skewness
    sk = skew(ret)
    results['Skewness'] = {
        'stat': sk, 'p': np.nan,
        'interpretation': (
            'Left-skewed (crash risk)' if sk < -0.5
            else 'Right-skewed (positive bias)' if sk > 0.5
            else 'Approximately symmetric'
        )
    }

    # Excess kurtosis
    kurt = kurtosis(ret, fisher=True)
    results['Excess Kurtosis'] = {
        'stat': kurt, 'p': np.nan,
        'interpretation': (
            'Heavy tails (leptokurtic) — extreme moves more likely' if kurt > 1
            else 'Thin tails (platykurtic)' if kurt < -1
            else 'Near-normal tails'
        )
    }

    # Ljung-Box test for autocorrelation in returns (lag 10)
    try:
        from statsmodels.stats.diagnostic import acorr_ljungbox
        lb = acorr_ljungbox(ret, lags=[10], return_df=True)
        lb_stat = lb['lb_stat'].values[0]
        lb_p = lb['lb_pvalue'].values[0]
        results['Ljung-Box(10)'] = {
            'stat': lb_stat, 'p': lb_p,
            'interpretation': (
                'Significant autocorrelation (predictable)' if lb_p < 0.05
                else 'No significant autocorrelation'
            )
        }
    except ImportError:
        pass

    # Ljung-Box on squared returns (ARCH effects)
    try:
        from statsmodels.stats.diagnostic import acorr_ljungbox
        lb2 = acorr_ljungbox(ret ** 2, lags=[10], return_df=True)
        lb2_stat = lb2['lb_stat'].values[0]
        lb2_p = lb2['lb_pvalue'].values[0]
        results['ARCH LB(10)'] = {
            'stat': lb2_stat, 'p': lb2_p,
            'interpretation': (
                'Volatility clustering present (GARCH justified)' if lb2_p < 0.05
                else 'No significant volatility clustering'
            )
        }
    except ImportError:
        pass

    # Variance ratio tests
    for lag in VR_LAGS:
        vr, z, p = variance_ratio_test(returns, lag)
        if not np.isnan(vr):
            results[f'VR({lag})'] = {
                'stat': vr, 'p': p,
                'interpretation': (
                    f'Momentum signal (VR={vr:.3f})' if vr > 1 and p < 0.05
                    else f'Mean-reversion signal (VR={vr:.3f})' if vr < 1 and p < 0.05
                    else f'Random walk (VR={vr:.3f})'
                )
            }

    return results


def classify_vol_regime(current_vol, vol_series):
    """
    Classify volatility regime by percentile rank.
    Returns (regime_label, percentile, color).
    """
    if len(vol_series.dropna()) < 20:
        return 'Insufficient Data', np.nan, THEME['text_dim']

    pct = sps.percentileofscore(vol_series.dropna(), current_vol)

    if pct <= 20:
        return 'LOW', pct, THEME['regime_low']
    elif pct <= 50:
        return 'MODERATE', pct, THEME['regime_mid']
    elif pct <= 80:
        return 'ELEVATED', pct, THEME['regime_high']
    else:
        return 'EXTREME', pct, THEME['regime_ext']


# ==============================================================================
#  SECTION 8: ANALYSIS ENGINE
# ==============================================================================

def run_volatility_analysis(data, ticker):
    """
    Master analysis function: runs every estimator, metric, and test.
    Returns a dict of all computed series and scalar results.
    """
    r = {}
    ret = data['Log_Return']
    o, h, l, c = data['Open'], data['High'], data['Low'], data['Close']

    # ── Core Estimators (3 windows) ──
    for label, win in [('Short', ROLLING_WINDOW_SHORT),
                       ('Medium', ROLLING_WINDOW_MEDIUM),
                       ('Long', ROLLING_WINDOW_LONG)]:
        r[f'RV_{label}']   = realized_volatility(ret, win)
        r[f'YZ_{label}']   = yang_zhang_volatility(o, h, l, c, win)
        r[f'DD_{label}']   = downside_deviation(ret, DOWNSIDE_MAR, win)

    # ── Range-based estimators (medium window) ──
    r['Parkinson'] = parkinson_volatility(h, l, ROLLING_WINDOW_MEDIUM)
    r['GK']        = garman_klass_volatility(o, h, l, c, ROLLING_WINDOW_MEDIUM)
    r['RS']        = rogers_satchell_volatility(o, h, l, c, ROLLING_WINDOW_MEDIUM)

    # ── Conditional models ──
    r['EWMA']  = ewma_volatility(ret, EWMA_LAMBDA)
    garch_series, garch_params = garch_volatility(ret, mode=GARCH_MODE)
    r['GARCH'] = garch_series
    r['GARCH_Params'] = garch_params

    # ── Higher-order / regime metrics ──
    r['VolOfVol']   = volatility_of_volatility(r['RV_Medium'], ROLLING_WINDOW_MEDIUM)
    r['ATR']        = calculate_atr(h, l, c, ATR_PERIOD)
    r['ATR_Pct']    = atr_percentile(r['ATR'], lookback=252)
    r['Efficiency'] = market_efficiency_ratio(r['RV_Medium'], r['GK'])

    # ── Hurst ──
    r['Hurst']         = rolling_hurst(ret, window=HURST_WINDOW, max_lag=50)
    r['Current_Hurst'] = calculate_hurst_exponent(ret, max_lag=HURST_MAX_LAG)

    # ── VaR / CVaR ──
    r['VaR'], r['CVaR'] = rolling_var_cvar(ret, window=VAR_WINDOW, confidence=VAR_CONFIDENCE)

    # ── Statistical Tests ──
    r['Distribution_Tests'] = return_distribution_tests(ret)

    # ── Regime Classification ──
    rv_med = r['RV_Medium'].dropna()
    if len(rv_med) > 0:
        r['Regime_Label'], r['Regime_Pctl'], r['Regime_Color'] = \
            classify_vol_regime(rv_med.iloc[-1], rv_med)
    else:
        r['Regime_Label'], r['Regime_Pctl'], r['Regime_Color'] = 'N/A', np.nan, THEME['text_dim']

    # ── Volatility Term Structure ──
    rv_s = r['RV_Short'].dropna()
    rv_l = r['RV_Long'].dropna()
    if len(rv_s) > 0 and len(rv_l) > 0:
        ratio = rv_s.iloc[-1] / rv_l.iloc[-1] if rv_l.iloc[-1] > 0 else np.nan
        if not np.isnan(ratio):
            if ratio > 1.15:
                r['Term_Structure'] = 'INVERTED (short > long — stress)'
            elif ratio < 0.85:
                r['Term_Structure'] = 'CONTANGO (short < long — calm)'
            else:
                r['Term_Structure'] = 'FLAT'
        else:
            r['Term_Structure'] = 'N/A'
    else:
        r['Term_Structure'] = 'N/A'

    return r


# ==============================================================================
#  SECTION 9: VISUALIZATION — HELPER UTILITIES
# ==============================================================================

def _format_ax(ax, title=None, ylabel=None, xlabel=None, legend=True, legend_loc='upper right'):
    """Apply consistent formatting to an axes object."""
    if title:
        ax.set_title(title, fontsize=THEME['font_title'], fontweight='bold',
                     color=THEME['text_bright'], pad=10, loc='left')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=THEME['font_label'])
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=THEME['font_label'])
    if legend:
        leg = ax.legend(loc=legend_loc, fontsize=THEME['font_legend'],
                        facecolor=THEME['panel'], edgecolor=THEME['grid'],
                        labelcolor=THEME['text'])
    ax.tick_params(colors=THEME['text_dim'], labelsize=THEME['font_tick'])
    for spine in ax.spines.values():
        spine.set_color(THEME['grid'])


def _annotate_box(ax, text, x=0.02, y=0.96, color=None, fontsize=None):
    """Place a styled annotation box on the axes."""
    ax.text(x, y, text, transform=ax.transAxes, fontsize=fontsize or THEME['font_annot'],
            fontweight='bold', color=color or THEME['text_bright'], va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=THEME['panel'],
                      edgecolor=color or THEME['grid'], alpha=0.95, linewidth=1.5))


def _vol_pct_formatter(x, _):
    """Format y-axis as percentage."""
    return f'{x:.0%}'


def _save_and_show(fig, filename, image_dir=None):
    """Display figure and save to images/volatility/ (F10)."""
    if SAVE_FIGURES:
        out_dir = Path(image_dir or VOLATILITY_IMAGE_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        try:
            fig.savefig(out_dir / filename, dpi=150, bbox_inches='tight',
                        facecolor=fig.get_facecolor(), edgecolor='none')
        except Exception as e:
            print(f"    ⚠ Failed to save {filename}: {e}")
    plt.show()
    plt.close(fig)


# ==============================================================================
#  SECTION 10: VISUALIZATION — PLOT FUNCTIONS
# ==============================================================================

def plot_price_with_vol(ax, data, results, ticker):
    """Price line + realized vol shaded area + EWMA overlay."""
    dates = data.index
    ax2 = ax.twinx()

    ax.plot(dates, data['Close'], color=THEME['text_bright'], linewidth=1.0, label='Close Price', zorder=3)
    ax.set_ylabel('Price', fontsize=THEME['font_label'], color=THEME['text'])
    ax.tick_params(axis='y', colors=THEME['text_dim'])

    rv = results['RV_Medium']
    ewma = results['EWMA']
    ax2.fill_between(dates, 0, rv, alpha=0.25, color=THEME['realized'], label=f'RV ({ROLLING_WINDOW_MEDIUM}d)')
    ax2.plot(dates, ewma, color=THEME['ewma'], linewidth=1.2, linestyle='--', label=f'EWMA (λ={EWMA_LAMBDA})')
    ax2.set_ylabel('Annualized Vol', fontsize=THEME['font_label'], color=THEME['realized'])
    ax2.tick_params(axis='y', colors=THEME['text_dim'])
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    safe_max = max(np.nanmax(rv.values), np.nanmax(ewma.values))
    ax2.set_ylim(0, safe_max * 1.4 if safe_max > 0 else 1)

    # Regime badge
    regime, pctl, rcolor = results.get('Regime_Label', ''), results.get('Regime_Pctl', np.nan), results.get('Regime_Color', THEME['text_dim'])
    _annotate_box(ax, f'Regime: {regime}  ({pctl:.0f}th %ile)', color=rcolor)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right',
               fontsize=THEME['font_legend'], facecolor=THEME['panel'],
               edgecolor=THEME['grid'], labelcolor=THEME['text'])
    ax.set_title(f'{ticker}  ·  Price & Volatility Overlay', fontsize=THEME['font_title'],
                 fontweight='bold', color=THEME['text_bright'], pad=10, loc='left')


def plot_estimator_comparison(ax, data, results, ticker):
    """All OHLC estimators on one panel for direct comparison."""
    dates = data.index
    for key, lbl, clr in [
        ('RV_Medium',  'Realized Vol',    THEME['realized']),
        ('Parkinson',  'Parkinson',       THEME['parkinson']),
        ('GK',         'Garman-Klass',    THEME['garman_klass']),
        ('RS',         'Rogers-Satchell', THEME['cyan']),
        ('YZ_Medium',  'Yang-Zhang',      THEME['yang_zhang']),
    ]:
        ax.plot(dates, results[key], label=lbl, linewidth=1.2, color=clr)

    mean_rv = results['RV_Medium'].mean()
    ax.axhline(mean_rv, color=THEME['text_dim'], linestyle=':', linewidth=0.8, alpha=0.6)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    _format_ax(ax, title=f'{ticker}  ·  Estimator Comparison ({ROLLING_WINDOW_MEDIUM}-Day)',
               ylabel='Ann. Volatility')


def plot_garch_ewma(ax, data, results, ticker):
    """GARCH vs EWMA — conditional volatility models."""
    dates = data.index
    ax.plot(dates, results['EWMA'], label=f'EWMA (λ={EWMA_LAMBDA})',
            linewidth=1.3, color=THEME['ewma'])
    gp = results.get('GARCH_Params', {})
    a, b = gp.get('alpha', np.nan), gp.get('beta', np.nan)
    persistence = gp.get('persistence', np.nan)
    half_life = gp.get('half_life', np.nan)
    garch_label = (f'GARCH(1,1) MLE (α={a:.3f}, β={b:.3f})'
                   if np.isfinite(a) and np.isfinite(b) else 'GARCH(1,1) MLE')
    ax.plot(dates, results['GARCH'], label=garch_label,
            linewidth=1.3, color=THEME['garch'], linestyle='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    if np.isfinite(persistence):
        annot = f'persistence = {persistence:.3f}, half-life = {half_life:.1f}d'
        color = THEME['yellow'] if persistence > 0.97 else THEME['green']
    else:
        annot = 'GARCH MLE did not converge'
        color = THEME['red']
    _annotate_box(ax, annot, color=color)
    _format_ax(ax, title=f'{ticker}  ·  EWMA vs GARCH(1,1) MLE', ylabel='Ann. Volatility')


def plot_yang_zhang_term_structure(ax, data, results, ticker):
    """Yang-Zhang across short / medium / long windows → vol term structure."""
    dates = data.index
    for key, lbl, clr, lw in [
        ('YZ_Short',  f'{ROLLING_WINDOW_SHORT}d',  THEME['blue'],   1.0),
        ('YZ_Medium', f'{ROLLING_WINDOW_MEDIUM}d',  THEME['yellow'], 1.8),
        ('YZ_Long',   f'{ROLLING_WINDOW_LONG}d',   THEME['red'],    1.3),
    ]:
        ax.plot(dates, results[key], label=lbl, linewidth=lw, color=clr)

    ax.fill_between(dates, results['YZ_Short'], results['YZ_Long'],
                    alpha=0.10, color=THEME['yellow'])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    ts = results.get('Term_Structure', 'N/A')
    ts_color = THEME['red'] if 'INVERTED' in ts else THEME['green'] if 'CONTANGO' in ts else THEME['text_dim']
    _annotate_box(ax, f'Term Structure: {ts}', color=ts_color)
    _format_ax(ax, title=f'{ticker}  ·  Yang-Zhang Term Structure', ylabel='Ann. Volatility')


def plot_downside_analysis(ax1, ax2, data, results, ticker):
    """Panel 1: total vol vs downside; Panel 2: downside ratio."""
    dates = data.index
    rv = results['RV_Medium']
    dd = results['DD_Medium']

    ax1.plot(dates, rv, label='Total Vol (RV)', linewidth=1.3, color=THEME['realized'])
    ax1.plot(dates, dd, label='Downside Dev', linewidth=1.3, color=THEME['downside'])
    ax1.fill_between(dates, dd, rv, alpha=0.15, color=THEME['green'], label='Upside Component')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    _format_ax(ax1, title=f'{ticker}  ·  Downside vs Total Volatility', ylabel='Ann. Volatility')

    ratio = dd / rv.replace(0, np.nan)
    ax2.plot(dates, ratio, linewidth=1.2, color=THEME['purple'])
    ax2.axhline(0.5, color=THEME['text_dim'], linestyle='--', linewidth=1.2, label='Symmetric (0.50)')
    ax2.axhline(0.7, color=THEME['red'], linestyle=':', linewidth=0.8, alpha=0.7, label='Skew threshold (0.70)')
    ax2.fill_between(dates, 0.5, ratio,
                     where=(ratio > 0.5), alpha=0.15, color=THEME['red'])
    ax2.fill_between(dates, 0.5, ratio,
                     where=(ratio < 0.5), alpha=0.15, color=THEME['green'])
    ax2.set_ylim(0, 1.1)
    _format_ax(ax2, title=f'{ticker}  ·  Downside Ratio (DD / RV)', ylabel='Ratio')


def plot_var_cvar(ax, data, results, ticker):
    """Rolling VaR and CVaR (Expected Shortfall)."""
    dates = data.index
    var_s = results['VaR']
    cvar_s = results['CVaR']

    ax.fill_between(dates, cvar_s, 0, alpha=0.20, color=THEME['red'], label=f'CVaR ({VAR_CONFIDENCE:.0%})')
    ax.plot(dates, var_s, linewidth=1.2, color=THEME['orange'], label=f'VaR ({VAR_CONFIDENCE:.0%})')
    ax.plot(dates, data['Log_Return'], linewidth=0.4, color=THEME['text_dim'], alpha=0.5, label='Daily Returns')

    # Mark breaches
    breaches = data['Log_Return'] < var_s
    if breaches.any():
        ax.scatter(dates[breaches], data['Log_Return'][breaches],
                   color=THEME['red'], s=12, zorder=5, alpha=0.7, label='VaR Breaches')
        breach_rate = breaches.sum() / len(data) * 100
        expected = (1 - VAR_CONFIDENCE) * 100
        _annotate_box(ax, f'Breach rate: {breach_rate:.1f}%  (expected: {expected:.0f}%)',
                      color=THEME['red'] if breach_rate > expected * 1.5 else THEME['green'])

    _format_ax(ax, title=f'{ticker}  ·  Rolling {VAR_CONFIDENCE:.0%} VaR & CVaR ({VAR_WINDOW}d)',
               ylabel='Daily Return')


def plot_vol_of_vol(ax, data, results, ticker):
    """Vol-of-vol with mean ± 2σ bands."""
    vov = results['VolOfVol'].dropna()
    if len(vov) == 0:
        ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center',
                transform=ax.transAxes, color=THEME['text_dim'])
        return

    ax.plot(vov.index, vov, linewidth=1.2, color=THEME['purple'])
    mu, sigma = vov.mean(), vov.std()
    ax.axhline(mu, color=THEME['text_dim'], linestyle='--', linewidth=0.8, label=f'Mean: {mu:.2%}')
    ax.axhline(mu + 2 * sigma, color=THEME['red'], linestyle=':', linewidth=0.8,
               label=f'+2σ: {mu + 2 * sigma:.2%}')
    ax.fill_between(vov.index, mu + 2 * sigma, vov,
                    where=(vov > mu + 2 * sigma), alpha=0.25, color=THEME['red'])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    _format_ax(ax, title=f'{ticker}  ·  Volatility of Volatility', ylabel='Vol-of-Vol')


def plot_atr_percentile(ax, data, results, ticker):
    """ATR percentile heatmap-style scatter."""
    atr_pct = results['ATR_Pct'].dropna()
    if len(atr_pct) == 0:
        ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center',
                transform=ax.transAxes, color=THEME['text_dim'])
        return

    colors = np.where(atr_pct < 25, THEME['regime_low'],
             np.where(atr_pct < 50, THEME['regime_mid'],
             np.where(atr_pct < 75, THEME['regime_high'], THEME['regime_ext'])))
    ax.scatter(atr_pct.index, atr_pct, c=colors, s=6, alpha=0.6, linewidths=0)

    for lvl, lbl, clr in [(80, '80th — Elevated', THEME['red']),
                           (20, '20th — Compressed', THEME['green']),
                           (50, '50th', THEME['text_dim'])]:
        ax.axhline(lvl, color=clr, linestyle='--' if lvl != 50 else ':', linewidth=0.8, label=lbl, alpha=0.8)

    ax.set_ylim(0, 100)
    _format_ax(ax, title=f'{ticker}  ·  ATR Percentile (1Y lookback)', ylabel='Percentile')


def plot_hurst(ax, results, ticker):
    """Rolling Hurst with regime shading."""
    hurst = results['Hurst'].dropna()
    h_now = results['Current_Hurst']

    if len(hurst) == 0:
        ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center',
                transform=ax.transAxes, color=THEME['text_dim'])
        return

    ax.plot(hurst.index, hurst.values, linewidth=1.2, color=THEME['cyan'])
    ax.axhline(0.5, color=THEME['text_bright'], linewidth=1.5, label='H=0.5 (Random Walk)')
    ax.axhline(0.6, color=THEME['yellow'], linestyle='--', linewidth=0.8, label='H=0.6 (Trending)')
    ax.axhline(0.4, color=THEME['blue'], linestyle='--', linewidth=0.8, label='H=0.4 (Mean-Rev)')
    ax.fill_between(hurst.index, 0.5, hurst,
                    where=(hurst > 0.5), alpha=0.15, color=THEME['yellow'])
    ax.fill_between(hurst.index, 0.5, hurst,
                    where=(hurst < 0.5), alpha=0.15, color=THEME['blue'])

    h_min = max(0, hurst.min() - 0.08)
    h_max = min(1, hurst.max() + 0.08)
    ax.set_ylim(max(0, min(h_min, 0.3)), min(1, max(h_max, 0.7)))

    if not np.isnan(h_now):
        regime = 'TRENDING' if h_now > 0.55 else 'MEAN-REV' if h_now < 0.45 else 'RANDOM'
        rclr = THEME['yellow'] if h_now > 0.55 else THEME['blue'] if h_now < 0.45 else THEME['text_dim']
        _annotate_box(ax, f'H = {h_now:.3f}  →  {regime}', color=rclr)

    _format_ax(ax, title=f'{ticker}  ·  Hurst Exponent ({HURST_WINDOW}d Rolling)',
               ylabel='Hurst (H)')


def plot_efficiency(ax, data, results, ticker):
    """Market efficiency ratio: RV / GK."""
    eff = results['Efficiency'].dropna()
    if len(eff) == 0:
        ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center',
                transform=ax.transAxes, color=THEME['text_dim'])
        return

    ax.plot(eff.index, eff, linewidth=1.2, color=THEME['green'])
    ax.axhline(1.0, color=THEME['text_bright'], linewidth=1.5, label='1.0 (Efficient)')
    ax.axhline(1.2, color=THEME['red'], linestyle='--', linewidth=0.8, label='>1.2 (Overnight gaps)')
    ax.axhline(0.8, color=THEME['blue'], linestyle='--', linewidth=0.8, label='<0.8 (Intraday)')
    ax.fill_between(eff.index, 1.0, eff, where=(eff > 1.0), alpha=0.12, color=THEME['red'])
    ax.fill_between(eff.index, 1.0, eff, where=(eff < 1.0), alpha=0.12, color=THEME['blue'])
    _format_ax(ax, title=f'{ticker}  ·  Market Efficiency Ratio (RV / GK)', ylabel='Ratio')


# ==============================================================================
#  SECTION 11: DASHBOARD ASSEMBLY
# ==============================================================================

def generate_asset_dashboard(data, results, ticker):
    """Generate the full multi-page dashboard for one asset."""

    # ═══════════════════════════════════════════════════════════════════════
    #  DASHBOARD 1 — CORE VOLATILITY (3 rows)
    # ═══════════════════════════════════════════════════════════════════════
    fig1, axes1 = plt.subplots(3, 1, figsize=(18, 13), sharex=True)
    fig1.suptitle(f'{ticker}  —  CORE VOLATILITY DASHBOARD',
                  fontsize=15, fontweight='bold', color=THEME['text_bright'], y=0.995)
    plot_price_with_vol(axes1[0], data, results, ticker)
    plot_estimator_comparison(axes1[1], data, results, ticker)
    plot_garch_ewma(axes1[2], data, results, ticker)
    fig1.tight_layout(rect=[0, 0, 1, 0.97])
    _save_and_show(fig1, f'{ticker}_dashboard1_core.png')

    # ═══════════════════════════════════════════════════════════════════════
    #  DASHBOARD 2 — TERM STRUCTURE & DOWNSIDE (3 rows)
    # ═══════════════════════════════════════════════════════════════════════
    fig2, axes2 = plt.subplots(3, 1, figsize=(18, 13), sharex=True)
    fig2.suptitle(f'{ticker}  —  TERM STRUCTURE & DOWNSIDE RISK',
                  fontsize=15, fontweight='bold', color=THEME['text_bright'], y=0.995)
    plot_yang_zhang_term_structure(axes2[0], data, results, ticker)
    plot_downside_analysis(axes2[1], axes2[2], data, results, ticker)
    fig2.tight_layout(rect=[0, 0, 1, 0.97])
    _save_and_show(fig2, f'{ticker}_dashboard2_termstructure.png')

    # ═══════════════════════════════════════════════════════════════════════
    #  DASHBOARD 3 — TAIL RISK & REGIME (2×2 grid)
    # ═══════════════════════════════════════════════════════════════════════
    fig3, axes3 = plt.subplots(2, 2, figsize=(18, 11))
    fig3.suptitle(f'{ticker}  —  TAIL RISK & REGIME INDICATORS',
                  fontsize=15, fontweight='bold', color=THEME['text_bright'], y=0.995)
    plot_var_cvar(axes3[0, 0], data, results, ticker)
    plot_vol_of_vol(axes3[0, 1], data, results, ticker)
    plot_hurst(axes3[1, 0], results, ticker)
    plot_efficiency(axes3[1, 1], data, results, ticker)
    fig3.tight_layout(rect=[0, 0, 1, 0.96])
    _save_and_show(fig3, f'{ticker}_dashboard3_tailrisk.png')

    # ═══════════════════════════════════════════════════════════════════════
    #  DASHBOARD 4 — ATR PERCENTILE (standalone for clarity)
    # ═══════════════════════════════════════════════════════════════════════
    fig4, ax4 = plt.subplots(figsize=(18, 4))
    fig4.suptitle(f'{ticker}  —  ATR PERCENTILE RANK',
                  fontsize=15, fontweight='bold', color=THEME['text_bright'], y=0.99)
    plot_atr_percentile(ax4, data, results, ticker)
    fig4.tight_layout(rect=[0, 0, 1, 0.92])
    _save_and_show(fig4, f'{ticker}_dashboard4_atr.png')


# ==============================================================================
#  SECTION 12: SUMMARY REPORT (CONSOLE)
# ==============================================================================

def print_asset_summary(data, results, ticker):
    """Print a comprehensive console report for one asset."""
    w = 78
    print(f"\n{'━' * w}")
    print(f"  {ticker}  —  VOLATILITY SUMMARY REPORT")
    print(f"  Data: {data.index[0].strftime('%Y-%m-%d')} → {data.index[-1].strftime('%Y-%m-%d')}  "
          f"({len(data)} trading days, ~{len(data)/252:.1f} years)")
    print(f"{'━' * w}")

    # ── Volatility Estimates ──
    print(f"\n  {'VOLATILITY ESTIMATES'}")
    print(f"  {'─' * 72}")
    print(f"  {'Estimator':<28} {'Current':>10} {'Mean':>10} {'Median':>10} {'Pctl':>8}")
    print(f"  {'─' * 72}")

    estimators = [
        ('Realized Vol (30d)',     'RV_Medium'),
        ('Parkinson (HL)',         'Parkinson'),
        ('Garman-Klass (OHLC)',    'GK'),
        ('Rogers-Satchell',        'RS'),
        ('Yang-Zhang (30d)',       'YZ_Medium'),
        ('EWMA (RiskMetrics)',     'EWMA'),
        ('GARCH(1,1)',             'GARCH'),
        ('Downside Dev (30d)',     'DD_Medium'),
    ]

    for name, key in estimators:
        s = results[key].dropna()
        if len(s) > 0:
            cur = s.iloc[-1]
            pct = sps.percentileofscore(s, cur)
            print(f"  {name:<28} {cur:>9.2%}  {s.mean():>9.2%}  {s.median():>9.2%}  {pct:>6.1f}%")

    # ── Regime ──
    print(f"\n  {'REGIME CLASSIFICATION'}")
    print(f"  {'─' * 72}")
    print(f"  Current regime:          {results.get('Regime_Label', 'N/A')} "
          f"({results.get('Regime_Pctl', 0):.0f}th percentile)")
    print(f"  Vol term structure:      {results.get('Term_Structure', 'N/A')}")

    h = results['Current_Hurst']
    h_str = f'{h:.3f}' if not np.isnan(h) else 'N/A'
    h_interp = ('TRENDING (momentum favored)' if h > 0.55
                else 'MEAN-REVERTING (reversion favored)' if h < 0.45
                else 'RANDOM WALK (efficient)') if not np.isnan(h) else 'N/A'
    print(f"  Hurst exponent:          {h_str}  →  {h_interp}")

    eff = results['Efficiency'].dropna()
    if len(eff) > 0:
        e = eff.iloc[-1]
        e_interp = ('Overnight gap-driven → use Yang-Zhang' if e > 1.1
                    else 'Intraday-driven → use Garman-Klass' if e < 0.9
                    else 'Efficient → any estimator works')
        print(f"  Efficiency ratio:        {e:.3f}  →  {e_interp}")

    # ── VaR ──
    var_s = results['VaR'].dropna()
    cvar_s = results['CVaR'].dropna()
    if len(var_s) > 0 and len(cvar_s) > 0:
        print(f"\n  {'TAIL RISK'}")
        print(f"  {'─' * 72}")
        print(f"  {VAR_CONFIDENCE:.0%} VaR (daily):          {var_s.iloc[-1]:>9.4f}  "
              f"(≈ {var_s.iloc[-1]*100:.2f}% worst-case daily loss)")
        print(f"  {VAR_CONFIDENCE:.0%} CVaR (daily):         {cvar_s.iloc[-1]:>9.4f}  "
              f"(≈ {cvar_s.iloc[-1]*100:.2f}% avg loss in tail)")

    # ── Statistical Tests ──
    tests = results.get('Distribution_Tests', {})
    if tests:
        print(f"\n  {'STATISTICAL TESTS'}")
        print(f"  {'─' * 72}")
        print(f"  {'Test':<22} {'Stat':>10} {'p-value':>10}  {'Interpretation'}")
        print(f"  {'─' * 72}")
        for tname, tvals in tests.items():
            stat_str = f"{tvals['stat']:.4f}" if not np.isnan(tvals['stat']) else 'N/A'
            p_str = f"{tvals['p']:.4f}" if not np.isnan(tvals['p']) else '—'
            sig = '***' if (not np.isnan(tvals['p']) and tvals['p'] < 0.01) \
                  else '**' if (not np.isnan(tvals['p']) and tvals['p'] < 0.05) \
                  else '*' if (not np.isnan(tvals['p']) and tvals['p'] < 0.10) else ''
            print(f"  {tname:<22} {stat_str:>10} {p_str:>10}{sig}  {tvals['interpretation']}")

    print(f"{'━' * w}\n")


# ==============================================================================


## Cross-asset comparison

Final-row regime/Hurst/skew/kurt table across all tickers, plus the
heavy-tail event diagnostic for BTAL and MNA. Failed tickers (delisted,
no data) surface as NaN rows with `(failed)` regime flags rather than
being silently dropped (F21).


In [ ]:
#  SECTION 13: CROSS-ASSET COMPARISON
# ==============================================================================

def generate_cross_asset_dashboard(all_data, all_results, failed_tickers=None):
    """Comparative dashboard across all assets. Failed tickers (F21) are
    represented as NaN-only rows so the reader can see which assets were
    requested but did not load, rather than silently dropping them."""
    failed_tickers = list(failed_tickers or [])
    if len(all_results) < 2:
        print("  (Skipping cross-asset dashboard — need ≥2 assets)")
        return

    tickers = list(all_results.keys())

    # F21: banner above the table so the reader sees data-quality issues
    # without having to scan the per-ticker logs. Failed tickers also
    # appear as a "Failed: N" row at the bottom of the text table; they
    # are deliberately *not* inserted into the chart DataFrame because
    # several downstream plots assume len(df) == len(tickers) and
    # broadcasting a NaN row against existing scalar arrays raises.
    if failed_tickers:
        n_total = len(tickers) + len(failed_tickers)
        print(f"\n  Note: {len(failed_tickers)} of {n_total} assets failed: "
              f"[{', '.join(failed_tickers)}]")

    # ── Build comparison table ──
    rows = []
    for t in tickers:
        r = all_results[t]
        d = all_data[t]
        rv = r['RV_Medium'].dropna()
        yz = r['YZ_Medium'].dropna()
        dd = r['DD_Medium'].dropna()
        ewma = r['EWMA'].dropna()

        row = {
            'Ticker':           t,
            'Days':             len(d),
            'RV_Current':       rv.iloc[-1] if len(rv) > 0 else np.nan,
            'RV_Mean':          rv.mean() if len(rv) > 0 else np.nan,
            'YZ_Current':       yz.iloc[-1] if len(yz) > 0 else np.nan,
            'EWMA_Current':     ewma.iloc[-1] if len(ewma) > 0 else np.nan,
            'DD_Current':       dd.iloc[-1] if len(dd) > 0 else np.nan,
            'Hurst':            r.get('Current_Hurst', np.nan),
            'Regime':           r.get('Regime_Label', ''),
            'Regime_Pctl':      r.get('Regime_Pctl', np.nan),
            'Term_Structure':   r.get('Term_Structure', ''),
            'Skew':             skew(d['Log_Return'].dropna().values),
            'Kurt':             kurtosis(d['Log_Return'].dropna().values, fisher=True),
        }
        rows.append(row)

    df = pd.DataFrame(rows).set_index('Ticker')

    # ═══════════════════════════════════════════════════════════════════════
    #  CHART 1: Current Vol by Estimator (grouped bar)
    # ═══════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('CROSS-ASSET VOLATILITY COMPARISON',
                 fontsize=16, fontweight='bold', color=THEME['text_bright'], y=0.995)

    ax = axes[0, 0]
    vol_cols = ['RV_Current', 'YZ_Current', 'EWMA_Current', 'DD_Current']
    vol_labels = ['Realized', 'Yang-Zhang', 'EWMA', 'Downside']
    vol_colors = [THEME['realized'], THEME['yang_zhang'], THEME['ewma'], THEME['downside']]
    x = np.arange(len(tickers))
    width = 0.18
    for i, (col, lbl, clr) in enumerate(zip(vol_cols, vol_labels, vol_colors)):
        vals = df[col].values
        ax.bar(x + i * width, vals, width, label=lbl, color=clr, alpha=0.85)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(tickers, rotation=45, ha='right', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_vol_pct_formatter))
    _format_ax(ax, title='Current Volatility by Estimator', ylabel='Ann. Vol')

    # ═══════════════════════════════════════════════════════════════════════
    #  CHART 2: Hurst Exponents
    # ═══════════════════════════════════════════════════════════════════════
    ax2 = axes[0, 1]
    h_vals = df['Hurst'].values
    h_colors = [THEME['yellow'] if v > 0.55 else THEME['blue'] if v < 0.45
                else THEME['text_dim'] for v in h_vals]
    ax2.barh(tickers, h_vals, color=h_colors, alpha=0.85)
    ax2.axvline(0.5, color=THEME['text_bright'], linewidth=1.5, linestyle='--', label='Random Walk')
    ax2.set_xlim(0.3, 0.7)
    _format_ax(ax2, title='Hurst Exponent (Current)', xlabel='H')

    # ═══════════════════════════════════════════════════════════════════════
    #  CHART 3: Skewness vs Kurtosis scatter
    # ═══════════════════════════════════════════════════════════════════════
    ax3 = axes[1, 0]
    ax3.scatter(df['Skew'], df['Kurt'], c=THEME['accent'], s=70, zorder=5, alpha=0.85)
    for t_idx, t_name in enumerate(tickers):
        ax3.annotate(t_name, (df['Skew'].iloc[t_idx], df['Kurt'].iloc[t_idx]),
                     fontsize=7, color=THEME['text'], xytext=(5, 5),
                     textcoords='offset points')
    ax3.axhline(0, color=THEME['text_dim'], linestyle=':', linewidth=0.7)
    ax3.axvline(0, color=THEME['text_dim'], linestyle=':', linewidth=0.7)
    _format_ax(ax3, title='Return Distribution: Skewness vs Excess Kurtosis',
               xlabel='Skewness', ylabel='Excess Kurtosis', legend=False)

    # ═══════════════════════════════════════════════════════════════════════
    #  CHART 4: Regime Percentile Rank
    # ═══════════════════════════════════════════════════════════════════════
    ax4 = axes[1, 1]
    pctl_vals = df['Regime_Pctl'].values
    pctl_colors = [THEME['regime_low'] if v <= 20 else THEME['regime_mid'] if v <= 50
                   else THEME['regime_high'] if v <= 80 else THEME['regime_ext'] for v in pctl_vals]
    ax4.barh(tickers, pctl_vals, color=pctl_colors, alpha=0.85)
    ax4.axvline(50, color=THEME['text_dim'], linestyle='--', linewidth=0.8)
    ax4.set_xlim(0, 100)
    _format_ax(ax4, title='Volatility Regime Percentile', xlabel='Percentile', legend=False)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    _save_and_show(fig, 'cross_asset_dashboard.png')

    # ── Print comparison table ──
    print(f"\n{'━' * 100}")
    print(f"{'CROSS-ASSET COMPARISON TABLE':^100}")
    print(f"{'━' * 100}")
    print(f"  {'Ticker':<8} {'Days':>5} {'RV Cur':>9} {'YZ Cur':>9} {'EWMA':>9} "
          f"{'DD':>9} {'Hurst':>7} {'Skew':>7} {'Kurt':>7} {'Regime':<12} {'Term Str'}")
    print(f"  {'─' * 95}")
    for _, row in df.iterrows():
        print(f"  {row.name:<8} {row['Days']:>5.0f} "
              f"{row['RV_Current']:>8.2%} {row['YZ_Current']:>8.2%} "
              f"{row['EWMA_Current']:>8.2%} {row['DD_Current']:>8.2%} "
              f"{row['Hurst']:>6.3f} {row['Skew']:>6.2f} {row['Kurt']:>6.2f} "
              f"{row['Regime']:<12} {row['Term_Structure']}")
    print(f"{'━' * 100}\n")

    # ── Heavy-tail diagnostic: top-5 |log_return| days for known
    # high-kurtosis tickers. Sanity-checks that the cross-asset table's
    # kurtosis figures are driven by real event days (e.g. COVID crash,
    # Pfizer vaccine announcement, deal breaks) rather than vendor data
    # artifacts. See PARTIAL_CHANGELOG.md for the BTAL 2015-04-29 print
    # error that triggered this diagnostic.
    HEAVY_TAIL_DIAGNOSTIC = ['BTAL', 'MNA']
    diagnostic_tickers = [t for t in HEAVY_TAIL_DIAGNOSTIC if t in all_data]
    if diagnostic_tickers:
        print(f"  {'─' * 95}")
        print(f"  Heavy-tail event diagnostic (top 5 |log return| days per ticker):")
        for tkr in diagnostic_tickers:
            r = all_data[tkr]['Log_Return'].dropna()
            if r.empty:
                continue
            top = r.reindex(r.abs().sort_values(ascending=False).index).head(5)
            print(f"    {tkr}:")
            for date, val in top.items():
                ds = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)[:10]
                print(f"      {ds}   log_r = {float(val):+.4f}  ({float(val)*100:+.2f}%)")
        print(f"  {'─' * 95}\n")


# ==============================================================================


## Run the full analysis


In [ ]:
#  SECTION 14: MAIN EXECUTION
# ==============================================================================

def run_analysis():
    """Main entry point."""
    print_header()

    all_data = {}
    all_results = {}
    failed = []

    for i, ticker in enumerate(TICKERS, 1):
        print(f"  [{i:>2}/{len(TICKERS)}]  {ticker}...")

        try:
            data = fetch_ohlc_data(ticker, START_DATE, END_DATE)
            if data.empty:
                failed.append(ticker)
                continue

            years = len(data) / 252
            print(f"         ✓ {len(data)} days loaded (~{years:.1f}y: "
                  f"{data.index[0].strftime('%Y-%m-%d')} → {data.index[-1].strftime('%Y-%m-%d')})")

            results = run_volatility_analysis(data, ticker)
            all_data[ticker] = data
            all_results[ticker] = results

            generate_asset_dashboard(data, results, ticker)

            if SHOW_SUMMARY:
                print_asset_summary(data, results, ticker)

        except Exception as e:
            print(f"         ✗ FAILED: {e}")
            traceback.print_exc()
            failed.append(ticker)

    # Cross-asset comparison
    if SHOW_CROSS_ASSET:
        generate_cross_asset_dashboard(all_data, all_results, failed_tickers=failed)

    # Final summary
    w = 74
    print(f"\n{'━' * w}")
    print(f"{'ANALYSIS COMPLETE':^{w}}")
    print(f"{'━' * w}")
    print(f"  Processed:  {len(all_results)} / {len(TICKERS)} assets")
    if failed:
        print(f"  Failed:     {', '.join(failed)}")
    print(f"{'━' * w}\n")

    return all_data, all_results


# ==============================================================================
#  EXECUTION
# ==============================================================================

if __name__ == "__main__":
    all_data, all_results = run_analysis()